#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.functions import col, lit, udf, sum, count, countDistinct, avg, max

#Reading Data From Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

#Some Data Transformations

In [0]:
df = (
    df.withColumn("prd_id",F.col("prd_id").cast("int"))
    .withColumn("prd_key",F.col("prd_key").cast("string"))
    .withColumn("prd_nm",F.col("prd_nm").cast("string"))
    .withColumn("prd_cost",F.col("prd_cost").cast("decimal(12,2)"))
    .withColumn("prd_line",F.col("prd_line").cast("string"))
    
)

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))
    

In [0]:
df = df.withColumn(
    "prd_line",
    F.when(F.col("prd_line") == "M" ,"Mountain")
    .when(F.col("prd_line") == "R", "Road")
    .when(F.col("prd_line") == "S", "Standard")
    .when(F.col("prd_line") == "T", "Touring")
    .otherwise("Unknown")
)

#Renaming the Columns

In [0]:
Rename_Map = {
    'prd_id':'product_id',
    'prd_key':'product_key',
    'prd_nm':'product_name',
    'prd_cost':'product_cost',
    'prd_line':'product_line',
    'prd_start_dt':'product_start_dt',
    'prd_end_dt':'product_end_dt'
}

In [0]:
for old_name , new_name in Rename_Map.items():
    df = df.withColumnRenamed(old_name,new_name)

In [0]:
df.display()

#Write into Silver Table

In [0]:
(
    df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("workspace.silver.crm_prd_info")
)